## Idea here is to make scripts that will be capable of taking bulk csv in a normalized format that will then be able to run and automatically make data into a wanddb friendly format and standardise it


To be added:
- fix below documentation according to changes
- sql query generation at the end
- sort out the doi collection cas right now new compounds just get none automatically (lotus did not have) (fix below)
- change the making working df for eventaul staging table section use an inner join the current implementation seems redundant by getting the compounds from wand that are in new compounds and then joining that 
- add functionality to the format checker function
- the check the error reporting system its not nca
- add result file number folded due to deduplication
## **Bulk compound normaliser**
#### **This compound normalizer should do the following:**
- get a list of compounds from wand and a list of new compounds
- check for missing smiles in new compounds, dropna and add to errors
- check new against wand and create a df that has all of the compounds not in wand and a df of compounds in wand
- **df of new compounds not in wand:**
    - generate canonical 2D and 3D SMILES from InChi with RDkit
    - fill na iupac using nispo
    - generate InChi key with RDkit
    - give wand compound_id starting after the last id from the wand compounds
    - assigns dois collections and cas as none and puts column names into db format
    - this df should be ready for a LOAD DATA LOCAL SQL
- **df of compounds in wand already:**
    - merges new compounds that are already in the df with the wand compounds that are in new compounds
    - adds synonyms from the new compounds to the existing synonyms with deduplication in a pipe seperated fromat
    - if existing wand name column name is empty incoming name fillsna
    - new name is removed if in synonyms
    - new iupac is added in a deduplicated manner to synonyms if there is an existing iupac
- **output:***
    - generate a text file with general info
    - an error file of compounds that had an issue during processing 
    - a new compounds csv that should be directly loadable into wand
    - a staging table for a bulk update

#### **Input requirements:**
- columns with multiple entries are pipe ('|') seperated
- **column headings new compounds:**
    - id (use an index or whatever it is for errors to be identifiable to compound later and cannot be NA)
    - new_id (new_id column must be named with source then _id i.e. lotus_id, wiki_id, can have multiple)
    - inchi
    - n_name
    - n_synonyms
    - n_iupac
    - copyable:) ['id','new_id','inchi','n_name','n_synonyms','n_iupac']
- **column headings wand compounds:**
    - compound_id
    - inchi
    - name
    - synonyms
    - iupac

In [ ]:
# wand compounds expected headings: all headings ie sum like ['compound_id','inchi','name','synonyms','iupac',dois,collections,coconut_id,cid,cas]
# new compounds expected headings ['id','new_id','inchi','n_name','n_synonyms','n_iupac', n_dois, n_collections,n_cids, n_cas]
# the existing id column must just be unique to the incoming compounds so it can be used to find compounds moved to the error file
# (new_id column must be named with source then _id i.e. lotus_id, wiki_id, can have multiple)
# all multi fields must be '|' seperated if have multi values
# i am moving all compounds that cant get an inchi or an inchikey to errors
# inchikeys are drawn from smiles as there can be issue converting inchi to mol and there is less of that with smiles
#//////////////////////////////////////////////////////inputs
wand_compounds_path = '/nlustre/users/nathanc/scratch/wand_test.csv'
new_compounds_path = '/nlustre/users/nathanc/scratch/coconut_test.csv'
staging_table='/nlustre/users/nathanc/scratch/staging.csv'
error_output_path = '/nlustre/users/nathanc/scratch/errors.csv'
output_path = '/nlustre/users/nathanc/scratch/compounds_for_db.csv'
report_path = '/nlustre/users/nathanc/scratch/report.txt'
#//////////////////////////////////////////////////////imports
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from nispo import mol_to_iupac
#//////////////////////////////////////////////////////data
wand_compounds = pd.read_csv(wand_compounds_path,low_memory=False)
new_compounds = pd.read_csv(new_compounds_path, low_memory= False)
origional_size = new_compounds.shape[0]
errors = pd.DataFrame(columns = ['id','type','error_msg'])
print('Files loaded', flush = True)
#//////////////////////////////////////////////////////functions
def get_normalized_smiles(smiles,id):
    global errors
    if smiles:
        try:
            mol = Chem.MolFromSmiles(smiles.strip())
            clean_mol = rdMolStandardize.Cleanup(mol)
            if clean_mol is not None:
                normalized_smiles = Chem.MolToSmiles(clean_mol,canonical=True, isomericSmiles=True)
                if normalized_smiles:
                    return normalized_smiles
                else: 
                    errors.loc[len(errors)] = err
                    err = [id,'smiles',f'Could not make cleaned mol into SMILES ({smiles})']
                    return pd.NA
            else:
                err = [id,'smiles',f'Could not clean molecule from SMILES ({smiles})']
                errors.loc[len(errors)] = err
                return None
        except Exception as e:
            err = [id,'smiles',str(e)]
            errors.loc[len(errors)] = err
            return None

def get_inchikey(smiles,id):
    global errors
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            key = Chem.MolToInchiKey(mol)
            return key
        else:
            err = [id,'normalised inchi',f'Could not convert normalized smiles ({smiles}) to inchi key']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [id,'normalised inchi',str(e)]
        errors.loc[len(errors)] = err
        return None

def get_iupac(smiles,id):
    global errors
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            iupac = mol_to_iupac(mol)
            if pd.notna(iupac):
                return iupac
            else:
                err = [id, 'normalised iupac', f'Could not generate IUPAC name from SMILES ({smiles})']
                errors.loc[len(errors)] = err
                return None
        else:
          err = [id,'normalised iupac',f'Could not convert SMILES ({smiles}) to iupac']
          errors.loc[len(errors)] = err
          return None  
    except Exception as e:
        err = [id,'normalised iupac',str(e)]
        errors.loc[len(errors)] = err
        return None 

def synonym_deduplicator(old,new):
    old = '' if pd.isna(old) else str(old)
    new = '' if pd.isna(new) else str(new)
    old = old.split('|')
    new = new.split('|')
    if '[]' in old:
        old.remove('[]')
    if '[]' in new:
        new.remove('[]')
    old = [ol.strip() for ol in old if ol.strip()]
    new = [ne.strip() for ne in new if ne.strip()]

    old_set = set(old)
    old_set = {ol.lower() for ol in old_set}
    for ne in new:
        if ne and ne.lower() not in old_set:
            old.append(ne)
            old_set.add(ne.lower())
    return '|'.join(old)

def name_syn_remover(name,synonyms):
    name = '' if pd.isna(name) else str(name)
    synonyms = '' if pd.isna(synonyms) else str(synonyms)
    name = name.strip()
    synonyms = synonyms.split('|')
    synonyms = [s.strip() for s in synonyms if s.strip()]
    synonyms = [s for s in synonyms if s.lower() != name.lower()]
    return '|'.join(synonyms)

def cid_with_progress(series,function):
    tot = len(series)
    count = 0
    def wrapped(value):
        nonlocal count
        count += 1
        if count % 100==0 or count == tot or count == 1:
            print(f'CID progress: {count} of {tot} at {count/tot*100:.2f}%', flush=True)
        result = function(value)
        return result
    return series.apply(wrapped)

def get_normalized_inchi(smiles, id):
    global errors
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            inchi = Chem.MolToInchi(mol)
            return inchi
        else:
            err = [id,'normalised smiles','Could not convert smiles to mol for inchi']
            errors.loc[len(errors)] = err
            return None
    except Exception as e:
        err = [id,'normalised smiles',str(e)]
        errors.loc[len(errors)] = err
        return None

def neat_list(strrr):
    out = []
    for strr in strrr:
        if pd.notna(strr):
            out.append(str(strr).strip())
    return '|'.join(out)  

def check_new_compounds_df_format(df):
    # need to add something that will make sure that all columns in wanddb that are also in the incoming they are labelled with the n_
    if new_compounds['id'].isna().any():
        raise Exception('There are missing ids in the new compounds and this column cannot have missing entries')    
    if new_compounds['smiles'].isna().any():
        raise Exception('There are missing SMILES in the new compounds and this column cannot have missing entries')

def wand_formatting(df,wand_cols):
    df = df.copy()
    df.drop(columns=['smiles'], inplace = True)
    df = df.rename(columns = {'norm_smiles': 'smiles'})
    cols = df.columns
    # rename columns with n_
    rename_dic = {}
    for col in cols:
        if 'n_' in col:
            rename_dic[col] = col.replace('n_','')
    df = df.rename(columns=rename_dic) 
    cols = df.columns
    # add wand columns not present
    for col in wand_cols:
        if col not in cols:
            df[col] = None
    # deal with new id columns
    new_order = wand_cols.copy()
    excluded = []
    for col in cols:
        if '_id' in col and col not in new_order:
            new_order.append(col)
        elif col not in new_order:
            excluded.append(col)
    new_cols = [col for col in new_order if col not in wand_cols]
    if len(new_cols) > 0:
        msg = f'New wand columns: {new_cols}\n'
    else:
        msg = ''
    if len(excluded) > 0:
        msg = msg + f'Columns in new compounds that were excluded: {excluded}\n'
    else:
        msg = msg + '' 
        
    df = df[new_order]
    return df, msg

def merge_syns(df):
    df = df.copy()
    cols = df.columns

    n_cols = []
    for col in cols:
        if col.startswith('n_'):
            n_cols.append(col)

    col_dic = {}
    for n_col in n_cols:
        no_n = n_col.replace('n_','')
        if no_n in cols:
            col_dic[n_col] = no_n

    for key, value in col_dic.items():
        df[value] = df.apply(lambda row: synonym_deduplicator(row[value],row[key]),axis = 1)

    return df

#//////////////////////////////////////////////////////new compounds
                        #////////////////////////////// format and completeness checks
check_new_compounds_df_format(new_compounds)
                        #////////////////////////////// create norm smiles column and normalize smiles for new compounds
print('Normalizing incoming SMILES', flush=True)
new_compounds['norm_smiles'] = new_compounds.apply(lambda row: get_normalized_smiles(row['smiles'],row['id']),axis = 1)
print('SMILES normalized', flush = True)
                        #//////////////////////////////check if new normalized smiles are missing or if are already in wanddb to seperate what is a new insert or what needs to update the existing entry (missing smiles go to error file and are removed)
print('Seperating existing and new compunds', flush = True)
wand_set = set(wand_compounds['smiles'].tolist())

if new_compounds['norm_smiles'].isna().any():
    missing_smiles = pd.DataFrame(columns=['id','type','error_msg'])
    missing_smiles['id'] = new_compounds['id'][new_compounds['smiles'].isna()]
    missing_smiles['type'] = 'smiles'
    missing_smiles['error_msg'] = 'missing smiles after normalization'
    errors = pd.concat([errors,missing_smiles], ignore_index=True)
new_compounds.dropna(subset=['norm_smiles'], inplace=True)

in_db = pd.DataFrame(columns=['id','type','error_msg'])
in_db['id'] = new_compounds[new_compounds['norm_smiles'].isin(wand_set)]['id'].to_list()
in_db['type'] = 'Existing compound'
in_db['error_msg'] = 'The normalized SMILES for this compound are already in wanddb it will be used to update wand if it has new ids or metadata'
errors = pd.concat([errors,in_db], ignore_index=True)
compounds_already_in_db = new_compounds[new_compounds['norm_smiles'].isin(wand_set)].copy()
new_compounds = new_compounds[~new_compounds['norm_smiles'].isin(wand_set)].copy()

print('Seperated old and new compounds', flush=True)
                        #//////////////////////////////deduplicating the new
print('Deduplicating', flush = True)
cols = new_compounds.columns.drop('norm_smiles')
pre_dup_size = len(new_compounds)
new_compounds = new_compounds.groupby('norm_smiles',as_index=False,sort=False).agg({column: neat_list for column in cols})
post_dup_size = len(new_compounds)
                        #//////////////////////////////deduplicating the existing
cols = compounds_already_in_db.columns.drop('norm_smiles')
new_compounds = new_compounds.groupby('norm_smiles',as_index=False,sort=False).agg({column: neat_list for column in cols})
print('Deduplication complete', flush= True)
                        #//////////////////////////////add normalized inchi
print('Adding normalized inchi')
new_compounds['norm_inchi'] = new_compounds.apply(lambda row: get_normalized_inchi(row['norm_smiles'],row['id']), axis = 1)
print('Normailzed inchi added')
                        #//////////////////////////////Add iupac
print('Adding iupac', flush=True)
new_compounds['n_iupac'] = new_compounds.apply( lambda row : (get_iupac(row['norm_smiles'], row['id']) if pd.isna(row['n_iupac']) else row['n_iupac']), axis = 1)
print('Iupac added', flush=True)
                        #//////////////////////////////Add inchikey
print('Adding inchikey', flush=True)
new_compounds['inchi_key'] = new_compounds.apply(lambda row : get_inchikey(row['norm_smiles'],row['id']), axis = 1)
print('DB formatting', flush=True)
                        #removing compounds that couldnt get inchi or inchikey
new_compounds.dropna(subset=['norm_inchi'], inplace=True)
new_compounds.dropna(subset=['inchi_key'], inplace=True)
                        #//////////////////////////////wand_ids
start_id = pd.to_numeric(wand_compounds['compound_id']).max()+1
new_compounds['compound_id'] = range(start_id, start_id+len(new_compounds))
                        #//////////////////////////////Formatting for wand
wand_cols = wand_compounds.columns
new_compounds, msg_from_new_column_addition = wand_formatting(new_compounds,wand_cols)
print('New Compounds formatted and ready for db', flush=True)
                        #//////////////////////////////saving compounds and intermediate error save
print('Saving new compounds', flush=True)
new_compounds.to_csv(output_path, index=False) 
errors.to_csv(error_output_path, index=False)
#//////////////////////////////////////////////////////update existing
print('Updating existing', flush=True)
                        #//////////////////////////////making working df for eventaul staging table
already_compounds = set(compounds_already_in_db['norm_smiles'].to_list())
staging_df = wand_compounds.loc[wand_compounds['smiles'].isin(already_compounds)].copy()
                        #//////////////////////////////updating names,iupac and synonyms
                        # removing uncessesary smiles and renaming the norm smiles
compounds_already_in_db.drop(columns=['smiles'])
compounds_already_in_db.rename(columns={'norm_smiles': 'smiles'})
try:
    staging_df = staging_df.merge(compounds_already_in_db, how='left',on='smiles', validate='1:1')
except Exception as e:
    print('There are duplicate normalised smiles in the wand or in the compounds that need to be added this should not be',flush=True)
    raise Exception(e)

if not staging_df.empty:
    staging_df['name'] = staging_df['name'].fillna(staging_df['n_name'])
    staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_name']), axis=1)

    staging_df['iupac'] = staging_df['iupac'].fillna(staging_df['n_iupac'])
    staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['iupac']), axis=1)
    staging_df.drop(columns = ['n_iupac'], inplace=True)

    staging_df['synonyms'] = staging_df.apply(lambda row: name_syn_remover(row['name'],row['synonyms']), axis=1)
    staging_df.drop(columns = ['n_name'], inplace=True)

    staging_df['synonyms'] = staging_df.apply(lambda row: synonym_deduplicator(row['synonyms'],row['n_synonyms']),axis=1)
    staging_df.drop(columns =['n_synonyms'], inplace=True)

    staging_df = merge_syns(staging_df)

print('Updated existing', flush=True)
#//////////////////////////////////////////////////////report
print('Generating report', flush=True)
new_size = new_compounds.shape[0]
errors_size = errors.shape[0]

items_removed = origional_size - new_size

error_types = errors['error_msg'].value_counts(dropna=False)

item = f"Compounds in starting file: {origional_size}\n\
         Numner of compounds pre-duplication: {pre_dup_size}\n\
         Number compounds post duplication: {post_dup_size}\n\
         New compounds standardised to add to the database: {new_size}\n\
         Removed compounds: {items_removed}\n\
         Errors: {errors_size}\n\
         Error types:\n\
         {error_types}\n\
         " + msg_from_new_column_addition

with open(report_path, 'w') as f:
    f.write(item)
#//////////////////////////////////////////////////////output
print('Saving', flush=True)
staging_df.to_csv(staging_table, index=False)
errors.to_csv(error_output_path, index=False)

Files loaded
Normalizing SMILES


SMILES normalized
Seperating existing and new compunds
Seperated onld and new
Deduplicating
Deduplication complete
Adding normalized inchi
Normailzed inchi added
Adding iupac
iupac added
Adding inchikey
DB formatting
New Compounds formatted and done
Updating existing
Updated existing
Generating report
Saving


# CID getter bulk 

In [ ]:
#!/bin/python3
# column names needed:['og_id','inchi','inchi_key','smiles','cid'] (can have the original ID it is not used) 
# writes names to read file
#module load python-3.9.5
#module load pubchempy
#//////////////////////////////////////////////////input
compounds_file = ''
err_out = ''
results_out = '' # txt for the stats of the run
#//////////////////////////////////////////////////imports
import time
import pandas as pd
import pubchempy as pcp
#//////////////////////////////////////////////////functions
def get_cid(input,type):
    if pd.isna(input):
        return f'no input for ({type})'
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower() or '503' in error_msg or 'too many requests or server too busy' in error_msg.lower():
                time.sleep(5)
            else:
                return str(e)
    return 'retry not sucessful'
       
def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out
#///////////////////////////////////////////////main
df = pd.read_csv(compounds_file)

err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
    count += 1
    if pd.notna(row.cid):
        if count % 100 == 0 or count == 1 or count == len(df):
                prog = progress_bar(count, length)
                print(prog, flush=True)
        continue
    inchi = row['inchi']
    inchi_key = row['inchi_key']
    smiles = row['smiles']

    cid = get_cid(inchi,'inchi')

    if isinstance(cid,int):
        df.at[idx, 'cid'] = cid
    else:
        err.append([inchi,cid])
        cid = get_cid(smiles,'smiles')
        if isinstance(cid,int):
            df.at[idx, 'cid'] = cid
        else:
            err.append([inchi,cid])
            cid = get_cid(inchi_key,'inchikey')
            if isinstance(cid,int):
                df.at[idx, 'cid'] = cid
            else:
                err.append([inchi,cid])
                df.at[idx, 'cid'] = pd.NA
    time.sleep(0.2)
    

    if count % 100 == 0 or count == 1 or count == len(df):
        prog = progress_bar(count, length)
        print(prog, flush=True)

    if count % 1000 == 0 or count == 1 or count == len(df):
        df.to_csv(compounds_file, index=False)
        error_df = pd.DataFrame(err, columns=['id', 'error'])
        error_df.to_csv(err_out,index=False)

#////////////////////////////////////////////////////outputs
df.to_csv(compounds_file,index=False)
error_df = pd.DataFrame(err, columns=['id', 'error'])
error_df.to_csv(err_out,index=False)

total = df.shape[0]
cids = df['cid'].notna().sum()
nothing = df['cid'].isna().sum()

errrs = error_df.shape[0]
messages = error_df['error'].value_counts(dropna=False).to_string()

results = f""" 
Total compounds: {total:,}
Compounds returned with cid: {cids:,}
Compounds no result: {nothing:,}
Percent sucess: {cids/total*100:.2f}% 
Total errors: {errrs:,} 
Messages: {messages}"""

with open(results_out, 'w') as f:
    f.write(results)

# CID getter one compound

In [ ]:
input = '' # string of smiles inchi or inchikey
type = ''# smiles/inchi/inchikey (shouldnt use type (reserved word) but oh well )

import pandas as pd
import pubchempy as pcp

def get_cid(input,type):
    if pd.isna(input):
        return f'no input for ({type})'
    for k in range(5):
        try:
            compound = pcp.get_compounds(input,type)
            if compound:
                cid = compound[0].cid
                return cid
            else:
                return None
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower() or '503' in error_msg or 'too many requests or server too busy' in error_msg.lower():
                time.sleep(5)
            else:
                return str(e)
    return 'retry not sucessful'

cid = get_cid(input,type)
print(cid)



only accepting full double barrel scientific names and variants no only genus or only species names
## **Bulk plant normalizer**
#### **This plant normalizer should do the following:**
- take in and format the wcvp
- take in a file retured from the checklist bank that has been deduplicated on the origional name
- keep only the GBIF name, ID, accepted name, accepted ID
- remove non-plants
- remove any genus or species only names (need both genus and species minimum)
- adding a cleaned name and matching the datasets based on that 
- fuzzy matching on the cleaned names using the fuzz.ratio scorer
- linking back to accepted ID
- collapsing into unified main ID with all else going to synonyms

In [ ]:
# the plants must go through the https://www.checklistbank.org/tools/name-match and that resulting file will be the new

# ///////////////////////////////////////////////////////inputs
wcvp_file = ''
new_checklisted_file = ''
staging_table= ''
error_output_path = ''
output_path = ''
report_path = ''
#////////////////////////////////////////////////////////imports
import pandas as pd
from rapidfuzz import process
#////////////////////////////////////////////////////////functions
def name_cleaner(name):
    if pd.isna(name):
        return pd.NA
    name = name.strip().lower().replace('-','').split()
    name = ' '.join(name)    
    return name
def is_incomplete_name(name):
    if pd.isna(name) or name == '' or not name.strip():
        return True
    else:
        name = name.strip()
        if len(name.split()) < 2:
            return True
        else: 
            return False
def fuzzy_match(og_name, og_id, name_id_dict):
    cutoff = 95
    result = process.extractOne(og_name,name_id_dict.keys())
    if result is None:
        err = err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to none of the wcvp']
        errors.loc[len(errors)] = err
        return pd.NA
    match, chance, _ = result
    if chance < cutoff:
        err = [og_id, 'match error', f'Failed direct match and fuzzy match with {og_name} matching to {match} at {chance}% chance']
        errors.loc[len(errors)] = err
        return pd.NA
    else:
        return name_id_dict.get(match)
    

#////////////////////////////////////////////////////////data
                                #////////////////////////load data
error_messages =''
wcvp = pd.read_csv(wcvp_file, sep='|')
new = pd.read_csv(new_checklisted_file)
errors = pd.DataFrame(columns=['id','error_stage','exact_error'])
                                #////////////////////////format wcvp
wcvp = wcvp.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'})
wcvp.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)

wcvp_name_check = wcvp[['scientific_name','taxonid']]
wcvp_name_check['clean_name'] = wcvp_name_check['scientific_name'].apply(name_cleaner)
wcvp_name_check_dic = {row.clean_name: row.taxonid for row in wcvp_name_check.itertuples()}
#////////////////////////////////////////////////////////normalize new plants
                                #////////////////////////drop unnecessary columns
new = new[['original_id','original_scientificName','matchType','matchIssues','ID','sceintificName','acceptedID','acceptedScientificName','kingdom']]
                                #////////////////////////removing non-plants
new = new.loc[new['kingdom'].eq('Plantae')].copy()
                                #////////////////////////removing incomplete names
new = new.loc[~new['original_scientificName'].apply(is_incomplete_name)].copy()
                                #////////////////////////adding clean name for direct matching
new['clean_name'] = new['original_scientificName'].apply(name_cleaner)
                                #////////////////////////checking for duplicates on clean name and removing if necessary
if new.duplicated(subset='clean_name').any():
   eagfdhskagfkhjasdgfkaj error_messages = error_messages + 'Once the names were cleaned for direct original name wcvp matching some of the cleaned names were duplicated the script keeps the first and removes the rest of the duplicates. Please ensure the original file and the checklist return file are deduplicated on the plant name. I this was done then the actual cleaning process leads to duplicates of names by removing "-" and whitespace as well as comparing only lowercase. This means redundancy was likely removed. The removed entries are in the error file. \n'
   eagfdhskagfkhjasdgfkaj duplicated = new.loc[new.duplicated(subset='clean_name',keep=False)].copy()
   eagfdhskagfkhjasdgfkaj duplicated_not_removed = duplicated.loc[duplicated.duplicated(subset='clean_name',keep='first')]['clean_name'].to_list()
   eagfdhskagfkhjasdgfkaj duplicated = duplicated.loc[~duplicated['clean_name'].isin(duplicated_not_removed)].copy()
   eagfdhskagfkhjasdgfkaj err = [[row.original_id,'Duplication',f'{row.original_name} cleaned to {row.clean_name} was duplicated after cleaning'] for row in duplicated.itertuples()]
   eagfdhskagfkhjasdgfkaj errors.loc[len(errors)] = err
   eagfdhskagfkhjasdgfkaj new = new.drop_duplicates(subset='clean_name',keep='first')
                                #////////////////////////matching cleaned names
new = new.merge(wcvp_name_check, how='left',on='clean_name', validate='many_to_one')
                                #////////////////////////fuzzy matching what is left
unmatched = new.loc[new['taxonid'].isna()].copy()
new['taxonid_fuzzy'] = unmatched.apply(lambda row: fuzzy_match(row.clean_name, row.original_id, wcvp_name_check_dic),axis=1)
                                #////////////////////////recording match numbers
new['match_type'] = pd.NA
new.loc[new['taxonid'].notna(),'match_type'] = 'direct_match'
new.loc[new['taxonid_fuzzy'].notna(),'match_type'] = 'fuzzy match' 
                                #////////////////////////removing all unmatched
#there is no need for an error handler here to add the dropped as they were added already during fuzzy match failure
new.dropna(subset='match_type',inplace=True)
                                #////////////////////////using taxon id to get no accepted info

                                #////////////////////////using accepted usage name id to get accepted info

                                #////////////////////////putting row non accepted into gbif and powo synonyms

                                #////////////////////////taking every accepted id that is taxon id and adding synonyms from where it is listed as accepted usage name id  

                                #////////////////////////formatting for DB with plant_id and correct columns

                                #remove entities that are weird matches
#////////////////////////////////////////////////////////plants already in db

#////////////////////////////////////////////////////////report
stats = f"""The number matched directly: {direct_matches:,} \n
            The number fuzzy matched: {fuzzy_matches:,}"""

item = error_messages + '\n' + stats
with open(report_path, 'w') as f:
    f.write(item)

#////////////////////////////////////////////////////////output